# Silver Layer

## Envirionment

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,LongType, ByteType
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
catalog_name = 'databricks-repo'
schema_name = 'silver'
volume_name = 'raw_enem'

volume_base_path = f'/Volumes/{catalog_name}/{schema_name}/{volume_name}'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

## Table Preview

In [0]:
schema_participantes = StructType(
    [
        StructField("NU_INSCRICAO", LongType(), False),
        StructField("NU_ANO", IntegerType(), False),
        StructField("TP_FAIXA_ETARIA", IntegerType(), False),
        StructField("TP_SEXO", StringType(), False),
        StructField("TP_ESTADO_CIVIL", IntegerType(), False),
        StructField("TP_COR_RACA", IntegerType(), False),
        StructField("TP_NACIONALIDADE", IntegerType(), False),
        StructField("TP_ST_CONCLUSAO", IntegerType(), False),
        StructField("TP_ANO_CONCLUIU", IntegerType(), False),
        StructField("TP_ENSINO", IntegerType(), True),
        StructField("IN_TREINEIRO", IntegerType(), False),
        StructField("CO_MUNICIPIO_PROVA", IntegerType(), False),
        StructField("NO_MUNICIPIO_PROVA", StringType(), False),
        StructField("CO_UF_PROVA", IntegerType(), False),
        StructField("SG_UF_PROVA", StringType(), False),
        StructField("Q001", StringType(), False),
        StructField("Q002", StringType(), False),
        StructField("Q003", StringType(), False),
        StructField("Q004", StringType(), False),
        StructField("Q005", StringType(), False),
        StructField("Q006", StringType(), False),
        StructField("Q007", StringType(), False),
        StructField("Q008", StringType(), False),
        StructField("Q009", StringType(), False),
        StructField("Q010", StringType(), False),
        StructField("Q011", StringType(), False),
        StructField("Q012", StringType(), False),
        StructField("Q013", StringType(), False),
        StructField("Q014", StringType(), False),
        StructField("Q015", StringType(), False),
        StructField("Q016", StringType(), False),
        StructField("Q017", StringType(), False),
        StructField("Q018", StringType(), False),
        StructField("Q019", StringType(), False),
        StructField("Q020", StringType(), False),
        StructField("Q021", StringType(), False),
        StructField("Q022", StringType(), False),
        StructField("Q023", StringType(), False),
    ]
)
df_participantes = spark.read.table(f"`{catalog_name}`.bronze.participantes")
df_participantes = df_participantes.filter(F.col("NU_INSCRICAO").isNotNull()).select(
    *[
        F.col(field.name).try_cast(field.dataType.simpleString())
        for field in schema_participantes.fields
    ]
)

df_participantes.show(10)

In [0]:
df_resultados.show(10)

## Schema Definition

In [0]:
schema_resultados = StructType(
    [
        StructField("NU_SEQUENCIAL", StringType()),
        StructField("NU_ANO", IntegerType()),
        StructField("CO_ESCOLA", IntegerType(), True),
        StructField("CO_MUNICIPIO_ESC", IntegerType()),
        StructField("NO_MUNICIPIO_ESC", StringType()),
        StructField("CO_UF_ESC", IntegerType()),
        StructField("SG_UF_ESC", StringType()),
        StructField("TP_DEPENDENCIA_ADM_ESC", IntegerType()),
        StructField("TP_LOCALIZACAO_ESC", IntegerType()),
        StructField("TP_SIT_FUNC_ESC", IntegerType()),
        StructField("CO_MUNICIPIO_PROVA", IntegerType()),
        StructField("NO_MUNICIPIO_PROVA", StringType()),
        StructField("CO_UF_PROVA", IntegerType()),
        StructField("SG_UF_PROVA", StringType()),
        StructField("TP_PRESENCA_CN", IntegerType()),
        StructField("TP_PRESENCA_CH", IntegerType()),
        StructField("TP_PRESENCA_LC", IntegerType()),
        StructField("TP_PRESENCA_MT", IntegerType()),
        StructField("CO_PROVA_CN", IntegerType()),
        StructField("CO_PROVA_CH", IntegerType()),
        StructField("CO_PROVA_LC", IntegerType()),
        StructField("CO_PROVA_MT", IntegerType()),
        StructField("NU_NOTA_CN", DoubleType()),
        StructField("NU_NOTA_CH", DoubleType()),
        StructField("NU_NOTA_LC", DoubleType()),
        StructField("NU_NOTA_MT", DoubleType()),
        StructField("TX_RESPOSTAS_CN", StringType()),
        StructField("TX_RESPOSTAS_CH", StringType()),
        StructField("TX_RESPOSTAS_LC", StringType()),
        StructField("TX_RESPOSTAS_MT", StringType()),
        StructField("TP_LINGUA", ByteType()),
        StructField("TX_GABARITO_CN", StringType()),
        StructField("TX_GABARITO_CH", StringType()),
        StructField("TX_GABARITO_LC", StringType()),
        StructField("TX_GABARITO_MT", StringType()),
        StructField("TP_STATUS_REDACAO", ByteType()),
        StructField("NU_NOTA_COMP1", IntegerType()),
        StructField("NU_NOTA_COMP2", IntegerType()),
        StructField("NU_NOTA_COMP3", IntegerType()),
        StructField("NU_NOTA_COMP4", IntegerType()),
        StructField("NU_NOTA_COMP5", IntegerType()),
        StructField("NU_NOTA_REDACAO", IntegerType()),
    ]
)
df_resultados = spark.read.table(f"`{catalog_name}`.bronze.resultados")
df_resultados = df_resultados.filter(F.col("NU_SEQUENCIAL").isNotNull()).select(
    *[
        F.col(field.name).try_cast(field.dataType.simpleString())
        for field in schema_resultados.fields
    ]
)
df_resultados.show(10)

In [0]:
df_itens_prova = spark.read.table(f'`{catalog_name}`.bronze.itens_prova')
df_itens_prova.show(10)

## Transformation

In [0]:
dicionario_dados = (
    spark.read.table("`databricks-repo`.bronze.dicionario_dados")
    .select(
        F.col("_c0").alias("nome"),
        F.col("_c1").alias("descricao"),
        F.col("_c2").alias("id_categoria"),
        F.col("_c3").alias("valor_categoria"),
    )
    .withColumn("index", F.monotonically_increasing_id())
    .filter(F.col("index") >= 4)
)

window_spec = Window.orderBy("index").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

dicionario_dados = dicionario_dados.withColumn(
    "nome", F.last("nome", ignorenulls=True).over(window_spec)
).withColumn("descricao", F.last("descricao", ignorenulls=True).over(window_spec))

display(dicionario_dados)

## Features Tables

In [0]:
class FeatureTables:
    def __init__(self, df: DataFrame):
        self._df = df

    def create_feature_table(self, table_name: str):
        """
        Creates a feature table in the catalog and schema specified in the notebook's configuration.
        """
        return self._df.filter(F.col("nome") == table_name).select(
            F.col("id_categoria").cast(IntegerType()),
            F.col("valor_categoria").cast(StringType()),
        )


f_tables = FeatureTables(dicionario_dados)

features_tables = {
    "ft_faixa_etaria": f_tables.create_feature_table("TP_FAIXA_ETARIA"),
    "ft_estado_civil": f_tables.create_feature_table("TP_ESTADO_CIVIL"),
    "ft_cor_raca": f_tables.create_feature_table("TP_COR_RACA"),
    "ft_nacionalidade": f_tables.create_feature_table("TP_NACIONALIDADE"),
    "ft_conclusao": f_tables.create_feature_table("TP_ST_CONCLUSAO"),
    "ft_ensino": f_tables.create_feature_table("TP_ENSINO"),
}

for _name, _table in features_tables.items():
    _table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"`{catalog_name}`.{schema_name}.{_name}"
    )

In [0]:
df_municipios = (
    df_resultados.select(
        F.col("CO_MUNICIPIO_PROVA").alias("id_muninicipio"),
        F.col("NO_MUNICIPIO_PROVA").alias("muninicipio"),
    )
    .distinct()
    .write.mode("overwrite")
    .saveAsTable(f"`{catalog_name}`.{schema_name}.municipio")
)

df_unidade_federativa = (
    df_resultados.select(
        F.col("CO_UF_PROVA").alias("id_uf"), F.col("SG_UF_PROVA").alias("uf_sigla")
    )
    .distinct()
    .write.mode("overwrite")
    .saveAsTable(f"`{catalog_name}`.{schema_name}.unidade_federativa")
)